# Concurrent Multi-Adapter Serving Infrastructures

## Overview

In multi-tenant SaaS environments, loading a separate full LLM instance per customer is cost-prohibitive. Serving hundreds of fine-tuned domains requires running one shared base model combined with dynamically loaded LoRA adapters.

## Incoming Mixed Concurrent Request Batch

```
┌───────────────────────────────────────────────────────────────────────────┐
│ Request 1 (Tenant A)  │  Request 2 (Tenant B)  │  Request 3 (Base Model)  │
└───────────┬───────────────────────┬───────────────────────┬───────────────┘
            │                       │                       │
            ▼                       ▼                       ▼
┌───────────────────────────────────────────────────────────────────────────┐
│                  Shared Base Model Weights W_0 (GPU VRAM)                 │
└───────────┬───────────────────────┬───────────────────────┬───────────────┘
            │                       │                       │
            ▼                       ▼                       ▼
  [ Compute (x1 * W_0) ]   [ Compute (x2 * W_0) ]   [ Compute (x3 * W_0) ]
            │                       │                       │
            ▼                       ▼                       │
┌───────────────────────┐ ┌───────────────────────┐         │
│  LoRA Adapter A Kern  │ │  LoRA Adapter B Kern  │         │
│ (x1 * alpha/r * B_A*A_A)│ │(x2 * alpha/r * B_B*A_B)│         │
└───────────┬───────────┘ └───────────┬───────────┘         │
            │                       │                       │
            ▼                       ▼                       ▼
      (+) Add Output          (+) Add Output          Base Output
```

## The Kernel Bottleneck: S-LoRA & Punica Kernels

Naively executing dynamic adapters in standard PyTorch loops causes extreme CUDA stream fragmentation and GPU stall conditions because each small adapter operation launches separate low-occupancy GPU kernels.

### Production Solution (S-LoRA / Punica / vLLM Multi-LoRA Execution)

**Segmented Gather/Scatter Kernels (BMM):**
* Group sequences in a batch by target adapter ID

**Unified Paged Adapter Memory:**
* Store all active adapter matrices $A_i, B_i$ in a unified non-contiguous GPU buffer (similar to PagedAttention)

**Custom Batch GEMM:**
* A specialized CUDA kernel computes batch matrix multiplications where each row $i$ in the input activation batch $X$ is multiplied by its corresponding adapter tensors $(A_{\text{adapter\_id}[i]}, B_{\text{adapter\_id}[i]})$ in a single fused GPU kernel call